In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import xml.etree.ElementTree as ET
from google.colab import drive

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("L2_Reports") \
    .getOrCreate()

In [ ]:
drive.mount('/content/drive')
POSTS_PATH = "/content/drive/MyDrive/big_data/data/posts_sample.xml"
LANGUAGES_PATH = "/content/drive/MyDrive/big_data/data/programming-languages.csv"

languages_df = spark.read.csv(LANGUAGES_PATH, header=True)
languages_df.show(10, truncate=False)
print(f"Всего языков: {languages_df.count()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
|ABSYS     |https:/

In [ ]:
languages_set = set(
    row['name'].strip().lower()
    for row in languages_df.collect()
    if row['name'] is not None
)

languages_broadcast = sc.broadcast(languages_set)

print(f"Количество уникальных языков: {len(languages_set)}")

Количество уникальных языков: 698


In [ ]:
def parse_row(line):
    try:
        elem = ET.fromstring(line.strip())
        creation_date = elem.get("CreationDate")
        tags_str = elem.get("Tags")

        if creation_date is None or tags_str is None:
            return []

        year = creation_date[:4]
        tags = re.findall(r'<([^>]+)>', tags_str)

        return [(year, tag.lower()) for tag in tags]
    except Exception:
        return []

In [ ]:
posts_rdd = sc.textFile(POSTS_PATH)
total_lines = posts_rdd.count()
row_rdd = posts_rdd.filter(lambda line: line.strip().startswith("<row"))

year_tag_rdd = row_rdd.flatMap(parse_row)

year_lang_rdd = year_tag_rdd.filter(
    lambda x: x[1] in languages_broadcast.value
)

year_lang_rdd.cache()

year_lang_count_rdd = year_lang_rdd.map(
    lambda x: ((x[0], x[1]), 1)
).reduceByKey(
    lambda a, b: a + b
).map(
    lambda x: (x[0][0], x[0][1], x[1])
)

print(f"Всего уникальных пар (год, язык): {year_lang_count_rdd.count()}")

Всего уникальных пар (год, язык): 468


In [ ]:
YEARS = [str(y) for y in range(2010, 2021)]
TOP_N = 10

results = []

for year in YEARS:
    top_langs = year_lang_count_rdd.filter(
        lambda x, y=year: x[0] == y
    ).sortBy(
        lambda x: -x[2]
    ).take(TOP_N)

    results.extend(top_langs)

print(f"Итого записей в отчёте: {len(results)}")

Итого записей в отчёте: 100


In [ ]:
schema = StructType([
    StructField("Year", StringType(), False),
    StructField("Language", StringType(), False),
    StructField("Count", IntegerType(), False)
])

report_df = spark.createDataFrame(results, schema=schema)

print("Топ 10 языков программирования по годам")
report_df.show(TOP_N * len(YEARS), truncate=False)

Топ 10 языков программирования по годам
+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2010|java       |52   |
|2010|php        |46   |
|2010|javascript |44   |
|2010|python     |26   |
|2010|objective-c|23   |
|2010|c          |20   |
|2010|ruby       |12   |
|2010|delphi     |8    |
|2010|r          |3    |
|2010|perl       |3    |
|2011|php        |102  |
|2011|java       |93   |
|2011|javascript |83   |
|2011|python     |37   |
|2011|objective-c|34   |
|2011|c          |24   |
|2011|ruby       |20   |
|2011|perl       |9    |
|2011|delphi     |8    |
|2011|bash       |7    |
|2012|php        |154  |
|2012|javascript |132  |
|2012|java       |124  |
|2012|python     |69   |
|2012|objective-c|45   |
|2012|c          |27   |
|2012|ruby       |27   |
|2012|bash       |10   |
|2012|r          |9    |
|2012|xpath      |6    |
|2013|javascript |198  |
|2013|php        |198  |
|2013|java       |194  |
|2013|python     |90   |
|2013|objective-c|40   |
|2013|c   

In [ ]:
report_df.write.parquet("/content/drive/MyDrive/big_data/L2 - Reports with Apache Spark/report.parquet")